ignore this part - conversion of pdf to image format (uploading to bluesky)

In [ ]:
FLYER_URI = "https://platform.foodhelpline.org/api/resources.pdf?lat=40.7598452&lng=-73.98547390000002&locationName=chelsea&flyerLang=en"

In [ ]:
import requests
import fitz  # PyMuPDF
import os

# Define the URL of the PDF
pdf_url = FLYER_URI

# Define an output folder
output_dir = "output_images"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

try:
    # 1. Download the PDF file content using requests
    response = requests.get(pdf_url, stream=True)
    response.raise_for_status()  # Check for bad response

    # 2. Open PDF from bytes using PyMuPDF
    pdf_document = fitz.open(stream=response.content, filetype="pdf")

    for i, page in enumerate(pdf_document):
        # Render page to a pixmap (image)
        pix = page.get_pixmap()
        # Save the image file
        output_path = os.path.join(output_dir, f"page_{i+1}.png")
        pix.save(output_path)
        print(f"Saved {output_path}")
    pdf_document.close()

except requests.exceptions.RequestException as e:
    print(f"Error downloading the PDF: {e}")
except Exception as e:
    print(f"An error occurred during conversion: {e}")

flyer creation stuff

In [ ]:
import warnings

warnings.simplefilter("default", DeprecationWarning)

In [ ]:
# Generate a PDF:
from fpdf import FPDF, FontFace

pdf = FPDF()
pdf.add_page()
pdf.set_font("helvetica", size=48)
pdf.cell(text="hello world")


pdf.add_page()
pdf.write_html(
    """
  <dl>
      <dt>Description title</dt>
      <dd>Description Detail</dd>
  </dl>
  <h1>Big title</h1>
  <section>
    <h2>Section title</h2>
    <p><b>Hello</b> world. <u>I am</u> <i>tired</i>.</p>
    <p><a href="https://github.com/py-pdf/fpdf2">py-pdf/fpdf2 GitHub repo</a></p>
    <p align="right">right aligned text</p>
    <p>i am a paragraph <br>in two parts.</p>
    <font color="#00ff00"><p>hello in green</p></font>
    <font size="7"><p>hello small</p></font>
    <font face="helvetica"><p>hello helvetica</p></font>
    <font face="times"><p>hello times</p></font>
  </section>
  <section>
    <h2>Other section title</h2>
    <ul type="circle">
      <li>unordered</li>
      <li>list</li>
      <li>items</li>
    </ul>
    <ol start="3" type="i">
      <li>ordered</li>
      <li>list</li>
      <li>items</li>
    </ol>
    <br>
    <br>
    <pre>i am preformatted text.</pre>
    <br>
    <blockquote>hello blockquote</blockquote>
    <table width="50%">
      <thead>
        <tr>
          <th width="30%">ID</th>
          <th width="70%">Name</th>
        </tr>
      </thead>
      <tbody>
        <tr>
          <td>1</td>
          <td>Alice</td>
        </tr>
        <tr>
          <td>2</td>
          <td>Bob</td>
        </tr>
      </tbody>
    </table>
  </section>
"""
)

pdf.add_page()
pdf.write_html(
    """
  <h1>Big title</h1>
  <section>
    <h2>Section title</h2>
    <p>Hello world!</p>
  </section>
""",
    tag_styles={
        "h1": FontFace(color="#948b8b", size_pt=32),
        "h2": FontFace(color="#948b8b", size_pt=24),
    },
)

pdf_bytes = pdf.output()

In [ ]:
# Display the PDF in the notebook by embedding it as HTML content:
WIDTH, HEIGHT = 800, 400
from base64 import b64encode
from IPython.display import display, HTML

base64_pdf = b64encode(pdf_bytes).decode("utf-8")
display(
    HTML(
        f'<embed height="{HEIGHT}" src="data:application/pdf;base64,{base64_pdf}" type="application/pdf" width="{WIDTH}"/>'
    )
)
display(
    HTML(
        f'<a download="fpdf2-demo.pdf" href="data:application/pdf;base64,{base64_pdf}">Click to download PDF</a>'
    )
)

In [ ]:
from pathlib import Path
from html import escape
from IPython.display import HTML, display

# Example payload using the Supabase campaign keys requested
campaign = {
    "title": "Chelsea Pantry Community Drive",
    "description": "Join neighbors and volunteers to distribute food and essentials to local families.",
    "date": "2026-03-22",
    "start_time": "10:00 AM",
    "end_time": "2:00 PM",
    "location": "Chelsea Community Center",
    "address": "123 W 26th St, New York, NY 10001",
}

required_keys = [
    "address",
    "date",
    "description",
    "start_time",
    "end_time",
    "title",
    "location",
]

missing = [k for k in required_keys if k not in campaign]
if missing:
    raise ValueError(f"Missing required campaign keys: {missing}")

safe = {k: escape(str(campaign.get(k, ""))) for k in required_keys}

flyer_html = f"""<!DOCTYPE html>
<html lang=\"en\">
<head>
  <meta charset=\"UTF-8\" />
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\" />
  <title>{safe['title']} - Promotional Flyer</title>
  <style>
    :root {{
      --bg-1: #f8fafc;
      --bg-2: #e2e8f0;
      --ink: #0f172a;
      --accent: #dc2626;
      --accent-2: #f59e0b;
    }}

    * {{ box-sizing: border-box; }}

    body {{
      margin: 0;
      font-family: \"Helvetica Neue\", Helvetica, Arial, sans-serif;
      color: var(--ink);
      background: radial-gradient(circle at top right, var(--bg-2), var(--bg-1));
      min-height: 100vh;
      display: grid;
      place-items: center;
      padding: 24px;
    }}

    .flyer {{
      width: min(900px, 100%);
      background: white;
      border: 4px solid var(--ink);
      border-radius: 18px;
      overflow: hidden;
      box-shadow: 0 16px 30px rgba(15, 23, 42, 0.18);
    }}

    .hero {{
      background: linear-gradient(120deg, var(--accent), var(--accent-2));
      color: white;
      padding: 28px 30px;
    }}

    .hero h1 {{
      margin: 0;
      font-size: clamp(1.8rem, 4vw, 2.8rem);
      line-height: 1.1;
      letter-spacing: 0.02em;
      text-transform: uppercase;
    }}

    .hero p {{
      margin: 10px 0 0;
      font-size: 1.02rem;
      max-width: 65ch;
    }}

    .content {{
      display: grid;
      gap: 18px;
      padding: 24px 30px 30px;
    }}

    .card {{
      border: 2px dashed #94a3b8;
      border-radius: 12px;
      padding: 14px 16px;
      background: #f8fafc;
    }}

    .meta {{
      display: grid;
      gap: 10px;
      grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
    }}

    .label {{
      font-size: 0.78rem;
      letter-spacing: 0.08em;
      font-weight: 700;
      color: #475569;
      text-transform: uppercase;
      margin-bottom: 4px;
    }}

    .value {{
      font-size: 1rem;
      font-weight: 600;
    }}

    .cta {{
      text-align: center;
      border: 2px solid var(--ink);
      border-radius: 12px;
      background: #fff7ed;
      padding: 16px;
      font-weight: 700;
      font-size: 1.05rem;
    }}
  </style>
</head>
<body>
  <article class=\"flyer\">
    <header class=\"hero\">
      <h1>{safe['title']}</h1>
      <p>{safe['description']}</p>
    </header>

    <section class=\"content\">
      <div class=\"meta\">
        <div class=\"card\">
          <div class=\"label\">Date</div>
          <div class=\"value\">{safe['date']}</div>
        </div>

        <div class=\"card\">
          <div class=\"label\">Time</div>
          <div class=\"value\">{safe['start_time']} - {safe['end_time']}</div>
        </div>

        <div class=\"card\">
          <div class=\"label\">Location</div>
          <div class=\"value\">{safe['location']}</div>
        </div>

        <div class=\"card\">
          <div class=\"label\">Address</div>
          <div class=\"value\">{safe['address']}</div>
        </div>
      </div>

      <div class=\"cta\">Volunteer. Share. Show up for your community.</div>
    </section>
  </article>
</body>
</html>
"""

output_html_path = Path("output_images") / "campaign_promotional_flyer.html"
output_html_path.parent.mkdir(parents=True, exist_ok=True)
output_html_path.write_text(flyer_html, encoding="utf-8")

print(f"Saved HTML flyer to: {output_html_path.resolve()}")
display(HTML(flyer_html))

In [ ]:
pdf2 = FPDF()
pdf2.add_page()

pdf2.write_html(flyer_html)
pdf2.output("output_images/campaign_flyer.pdf")
print("Saved PDF flyer to: output_images/campaign_flyer.pdf")

In [22]:
from pathlib import Path
from html import escape
from fpdf import FPDF, FontFace

required_keys = [
    "address",
    "date",
    "description",
    "start_time",
    "end_time",
    "title",
    "location",
]

missing = [k for k in required_keys if k not in campaign]
if missing:
    raise ValueError(f"Missing required campaign keys: {missing}")

safe = {k: escape(str(campaign.get(k, ""))) for k in required_keys}

flyer_html_for_pdf = f"""
<h1 align="center">{safe['title']}</h1>
<h2 align="center">Community Outreach Event</h2>
<p align="center"><strong>{safe['description']}</strong></p>

<p><strong>DATE:</strong> {safe['date']}</p>
<p><strong>TIME:</strong> {safe['start_time']} - {safe['end_time']}</p>
<p><strong>LOCATION:</strong> {safe['location']}</p>
<p><strong>ADDRESS:</strong> {safe['address']}</p>

<h2 align="center">Volunteer. Share. Show up for your community.</h2>
"""

pdf2 = FPDF(format="A4")
pdf2.set_auto_page_break(auto=True, margin=16)
pdf2.add_page()

pdf2.write_html(
    flyer_html_for_pdf,
    tag_styles={
        "h1": FontFace(emphasis="B", size_pt=44, color="#0B2545"),
        "h2": FontFace(emphasis="B", size_pt=28, color="#1D4E89"),
        "p": FontFace(size_pt=20, color="#0F172A"),
        "strong": FontFace(emphasis="B", size_pt=21, color="#0B2545"),
    },
)

output_pdf_path = Path("output_images") / "campaign_promotional_flyer_large_text.pdf"
output_pdf_path.parent.mkdir(parents=True, exist_ok=True)
pdf2.output(str(output_pdf_path))

print(f"Saved PDF flyer to: {output_pdf_path.resolve()}")

Saved PDF flyer to: /Users/chris/Documents/GitHub/TrackATeam4Project/backend/output_images/campaign_promotional_flyer_large_text.pdf
